From Andy:

The first tranche of gridded molefraction files has been recovered and transferred to MSU.

Files are in /work/noaa/co2/andy/Projects/WOMBAT/wombat-v3-forward/3b_transport_tm5/intermediates/runs, for runs r0001 through r0207. TM5 stores these in daily files in output directories named after the job step start time. These start times are spaced by 4 weeks. So:

r0208/output/20200207/molefrac_glb3x2_202002250000_202002260000.nc

is associated with a jobstep that started 2020-02-07, and is a daily average from 0Z on 2020-02-25 to 0Z on 2020-02-26. It has tracers s0016048 through s0017107. These tracer names are decoded in:

/work/noaa/co2/andy/Projects/WOMBAT/wombat-v3-forward/3b_transport_tm5/intermediates/runs/tm5_mapping.csv.

In [1]:
import pdb
import glob
import pandas as pd
import numpy as np
import xarray as xr
import inversion_tools
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

globf = lambda x: [s.split('/')[-1] for s in glob.glob(x)]
pdir  = '/work/noaa/co2/jhollo/processed_transport_data/tm5_tmp'

In [2]:
tm5dir = '/work/noaa/co2/andy/Projects/WOMBAT/wombat-v3-forward/3b_transport_tm5/intermediates'
mapping = pd.read_csv(f'{tm5dir}/runs/tm5_mapping.csv')

In [3]:
dpd = 28 # days-per-directory expected. This shouldbe 4 weeks according to Andy, or 28 days

fails = 0
completed_runs = []
bad_runs       = []

for i,row in enumerate(np.arange(len(mapping))):
    cfails = fails
    ffails, mffails = 0, 0
    
    run      = mapping.iloc[i]['run']
    start    = mapping.iloc[i]['start_time']
    end      = mapping.iloc[i]['end_time']
    species  = mapping.iloc[i]['species']
    if(run in completed_runs or run in bad_runs):
        continue

    # find expected number of days
    start = np.datetime64(start.split()[0])
    end   = np.datetime64(end.split()[0])
    ndays = (end - start).astype('timedelta64[D]').astype(int)

    # find all subdirs in this run dir
    run_dirs = glob.glob(f'{tm5dir}/runs/{run}/output/*')
    nrundirs = len(run_dirs)
    nrundays = nrundirs * dpd

    # step 1: verify number of subdirs
    if(nrundays < ndays):
        print(f'ndays check FAILED on run: {run}: expected at least {ndays} days of data, found only {nrundays}')
        fails += 1

    # step 2: number of files in each dir
    for j,rdir in enumerate(run_dirs):
        
        flux_files = glob.glob(f'{rdir}/flux1x1_*')
        nf = len(flux_files)
        if(nf != dpd * 24):
            #print(f'flux files check FAILED on run:{run}/{rdir.split('/')[-1]}, species:{species}: expected {dpd*24} hourly flux files, found {nf}')
            ffails += 1
        mf_files   = glob.glob(f'{rdir}/molefrac_*')
        nmf = len(mf_files)
        if(nmf != dpd * (24/3)):
            #print(f'mf files check FAILED on run:{run}/{rdir.split('/')[-1]}, species:{species}: expected {int(dpd*(24/3))} 3-hourly flux files, found {nmf}')
            mffails += 1
        if(mffails + ffails > 0):
            fails += 1

    if(fails == cfails):
        print(f'------ run {run} verified!')
        completed_runs += [run]
    else:
        print(f'run {run} FAILED with {ffails}/{nrundirs} bad flux dirs, {mffails}/{nrundirs} bad mf dirs')
        bad_runs += [run]

print('-----------------------------------')
print(f'completed runs: {completed_runs}')
print(f'bad runs: {bad_runs}')
        

------ run r0001 verified!
------ run r0002 verified!
------ run r0003 verified!
run r0004 FAILED with 123/125 bad flux dirs, 111/125 bad mf dirs
------ run r0005 verified!
------ run r0006 verified!
------ run r0007 verified!
------ run r0008 verified!
------ run r0009 verified!
------ run r0010 verified!
------ run r0011 verified!
------ run r0012 verified!
------ run r0013 verified!
------ run r0014 verified!
------ run r0015 verified!
------ run r0016 verified!
------ run r0017 verified!
------ run r0018 verified!
------ run r0019 verified!
------ run r0020 verified!
------ run r0021 verified!
------ run r0022 verified!
------ run r0023 verified!
------ run r0024 verified!
------ run r0025 verified!
------ run r0026 verified!
run r0027 FAILED with 123/125 bad flux dirs, 111/125 bad mf dirs
run r0028 FAILED with 123/125 bad flux dirs, 111/125 bad mf dirs
------ run r0029 verified!
------ run r0030 verified!
------ run r0031 verified!
------ run r0032 verified!
------ run r0033 verif